# SHIPIT Agent Test Drive

Use this notebook to validate provider setup, inspect the built-in runtime, and run a local multi-tool smoke test.

Main workflow:
- load `.env`
- build the demo agent
- run `agent.doctor()`
- execute a normal prompt
- inspect streaming events
- use `agent.chat_session(...)`
- inspect websocket and SSE packets
- review the attached tools


In [1]:
from pathlib import Path
import sys, importlib.util
from IPython.display import Markdown, display

# Locate repo root by walking up until we find examples/run_multi_tool_agent.py
def _find_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'examples' / 'run_multi_tool_agent.py').exists():
            return candidate
    raise RuntimeError(f'Could not locate shipit_agent repo root from {start}')

ROOT = _find_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Load the example module directly by file path (examples/ has no __init__.py,
# so a normal `from examples.run_multi_tool_agent import ...` can fail).
_spec = importlib.util.spec_from_file_location(
    'run_multi_tool_agent', ROOT / 'examples' / 'run_multi_tool_agent.py'
)
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
build_demo_agent = _mod.build_demo_agent
build_llm_from_env = _mod.build_llm_from_env
load_env_file = _mod.load_env_file

load_env_file(ROOT / '.env')
ROOT

PosixPath('/Users/rahulraj/Documents/MYWORK/ai_developer/shipit_agent')

In [2]:
import os

# Custom provider selection — reads from .env (SHIPIT_LLM_PROVIDER), falls back to bedrock.
# Supported: bedrock, openai, anthropic, gemini, groq, together, ollama
provider = os.environ.get('SHIPIT_LLM_PROVIDER', 'bedrock')
print(f'Using provider: {provider}')

llm = build_llm_from_env(provider)
agent = build_demo_agent(llm=llm, workspace_root=str(ROOT / '.shipit_notebook_workspace'))

report = agent.doctor()
display(Markdown(report.to_markdown()))
report.to_dict()

Using provider: bedrock


# SHIPIT Agent Doctor Report

- Passed: yes
- Checks: 6
- Failures: 0
- Warnings: 2

## PASS llm_provider
bedrock configuration looks ready.
- class_name: BedrockChatLLM
- model: bedrock/openai.gpt-oss-120b-1:0

## PASS tools
Tool registry looks consistent.
- tool_count: 28
- tools: web_search, open_url, playwright_browse, ask_user, human_review, memory, plan_task, decompose_problem, synthesize_evidence, decision_matrix, build_prompt, verify_output ...

## WARN mcps
No MCP servers are attached.
- mcp_count: 0

## PASS stores
Memory, session, and trace stores are configured.
- memory_store: FileMemoryStore
- session_id: demo-session
- session_store: FileSessionStore
- trace_id: demo-trace
- trace_store: FileTraceStore

## WARN connectors
Connector tools are attached but no credential store is configured.
- connector_count: 9

## PASS runtime_limits
Runtime iteration budget looks reasonable.
- max_iterations: 4

{'passed': True,
 'check_count': 6,
 'failure_count': 0,
 'warning_count': 2,
 'checks': [{'name': 'llm_provider',
   'status': 'pass',
   'message': 'bedrock configuration looks ready.',
   'details': {'class_name': 'BedrockChatLLM',
    'model': 'bedrock/openai.gpt-oss-120b-1:0'}},
  {'name': 'tools',
   'status': 'pass',
   'message': 'Tool registry looks consistent.',
   'details': {'tool_count': 28,
    'tools': 'web_search, open_url, playwright_browse, ask_user, human_review, memory, plan_task, decompose_problem, synthesize_evidence, decision_matrix, build_prompt, verify_output ...'}},
  {'name': 'mcps',
   'status': 'warn',
   'message': 'No MCP servers are attached.',
   'details': {'mcp_count': 0}},
  {'name': 'stores',
   'status': 'pass',
   'message': 'Memory, session, and trace stores are configured.',
   'details': {'memory_store': 'FileMemoryStore',
    'session_store': 'FileSessionStore',
    'trace_store': 'FileTraceStore',
    'session_id': 'demo-session',
    'trace_

## Chat Session Wrapper

Use `agent.chat_session(...)` when you want a stable chat-style wrapper around one conversation. This is the right layer for app integrations.


In [3]:
session = agent.chat_session(session_id='notebook-chat', trace_id='notebook-chat-trace')
session_result = session.send('Summarize this project as if this were a product chat session.')
print('CHAT OUTPUT:\n', session_result.output)
len(session.history())

CHAT OUTPUT:
 Sure thing! Could you share the project description or the key details you’d like summarized? That’ll help me craft a concise product‑chat style overview for you.


5

In [4]:
# Quick non-streaming sanity run — shows final output + first few tool results.
result = agent.run('Use the available tools and summarize what this project can do.')
print('FINAL OUTPUT:\n', result.output, '\n')
print('TOOL RESULTS:')
for tr in result.tool_results[:8]:
    print(f'  • {tr.name}: {tr.output[:140]}')

FINAL OUTPUT:
  

TOOL RESULTS:
  • workspace_files: sessions/demo-session.json
sessions/notebook-chat.json
  • workspace_files: {
  "session_id": "demo-session",
  "messages": [
    {
      "role": "system",
      "content": "You are Shipit, a capable general-purpose 
  • workspace_files: memory.json
sessions
traces
  • workspace_files: memory.json
sessions
traces


## WebSocket Packets

Frontend apps often want JSON-friendly packets. `stream_packets(..., transport='websocket')` returns packet dictionaries that are easy to push over a socket.


In [5]:
ws_packets = list(session.stream_packets(
    'Plan the work and show packet-style updates.',
    transport='websocket',
))
ws_packets[:5]


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.



BadRequestError: litellm.BadRequestError: BedrockException - {"message":"The number of toolResult blocks at messages.6.content exceeds the number of toolUse blocks of previous turn."}

## SSE Packets

For server-sent events, `stream_packets(..., transport='sse')` returns text packets ready to stream from a backend endpoint.


In [6]:
sse_packets = list(session.stream_packets(
    'Explain the runtime in SSE packet form.',
    transport='sse',
))
sse_packets[:3]

['event: run_started\ndata: {"message": "Agent run started", "payload": {"prompt": "Explain the runtime in SSE packet form."}, "type": "run_started"}\n\n',
 'event: step_started\ndata: {"message": "LLM completion started", "payload": {"iteration": 1, "tool_count": 28}, "type": "step_started"}\n\n',
 'event: run_completed\ndata: {"message": "Agent run completed", "payload": {"output": "```\\nevent: intro\\ndata: {\\"title\\":\\"Shipit Runtime Overview\\",\\"summary\\":\\"A general\\u2011purpose agent runtime that orchestrates tool usage, planning, and execution to solve user tasks end\\u2011to\\u2011end.\\"}\\n\\nevent: core_behavior\\ndata: {\\"principles\\":[\\"Accurate\\",\\"Direct\\",\\"Execution\\u2011oriented\\"],\\"actions\\":[\\"Solve tasks completely\\",\\"Prefer fresh data via tools\\",\\"Use smallest correct tool\\",\\"Plan for complex tasks\\",\\"Verify important results\\"]}\\n\\nevent: tool_behavior\\ndata: {\\"process\\":\\"Read tool description \\u2192 Choose minimal app

In [7]:
import json, textwrap

def _short(val, n=400):
    s = val if isinstance(val, str) else json.dumps(val, default=str)
    return textwrap.shorten(s.replace('\n', ' '), width=n, placeholder=' …')

def pretty_event(ev):
    t, msg, p = ev.type, ev.message or '', ev.payload or {}
    itr = p.get('iteration')
    tag = f'[{t}]' + (f' (iter={itr})' if itr is not None else '')

    if t == 'run_started':
        return f'{tag} prompt="{_short(p.get("prompt",""),160)}"'
    if t == 'planning_started':
        return f'{tag} planning: "{_short(p.get("prompt",""),160)}"'
    if t == 'planning_completed':
        return f'{tag} plan → {_short(p.get("output",""))}'
    if t == 'step_started':
        return f'{tag} tools_available={p.get("tool_count")}'
    if t == 'tool_called':
        return f'{tag} → {msg}\n       args={_short(p.get("arguments", {}))}'
    if t == 'tool_completed':
        return f'{tag} ✓ {msg}\n       output={_short(p.get("output",""))}'
    if t == 'tool_failed':
        return f'{tag} ✗ {msg} :: {_short(p.get("error",""))}'
    if t == 'tool_retry':
        return f'{tag} ↻ attempt={p.get("attempt")} {_short(p.get("error",""))}'
    if t == 'llm_retry':
        return f'{tag} ↻ llm attempt={p.get("attempt")} :: {_short(p.get("error",""))}'
    if t == 'mcp_attached':
        return f'{tag} 🔌 mcp server={p.get("server")}'
    if t == 'interactive_request':
        return f'{tag} ? kind={p.get("kind")} payload={_short(p.get("payload", {}))}'
    if t == 'run_completed':
        return f'{tag} ■ {_short(p.get("output",""))}'
    return f'{tag} {msg} payload={_short(p)}'


def stream_and_print(prompt: str, title: str):
    print(f'\n{"="*80}\n▶ {title}\n  PROMPT: {prompt}\n{"="*80}')
    collected, tools_used = [], []
    for ev in agent.stream(prompt):
        collected.append(ev)
        print(pretty_event(ev))
        if ev.type == 'tool_called':
            tools_used.append(ev.message)
    print(f'\n  ↳ {len(collected)} events, tools invoked: {tools_used}')
    return collected


# Real-tool streaming tests — force the agent to actually hit web_search, open_url,
# planner, workspace_files, code_execution, and memory so you can watch live packets.
run1 = stream_and_print(
    'Use the web_search tool to look up "Claude Opus 4.6 release notes", '
    'then use open_url on the top result and summarize key capabilities in 5 bullets.',
    'TEST 1: web_search + open_url',
)

run2 = stream_and_print(
    'Use the planner tool to draft a 4-step plan for shipping a Python CLI, '
    'then use the workspace_files tool to list files under the workspace root.',
    'TEST 2: planner + workspace_files',
)

run3 = stream_and_print(
    'Use code_execution to compute the first 10 Fibonacci numbers, '
    'then use the memory tool to store the result under key "fib10".',
    'TEST 3: code_execution + memory',
)

print(f'\nTotal events across 3 runs: {len(run1)+len(run2)+len(run3)}')


▶ TEST 1: web_search + open_url
  PROMPT: Use the web_search tool to look up "Claude Opus 4.6 release notes", then use open_url on the top result and summarize key capabilities in 5 bullets.


Error: BrowserType.launch: Executable doesn't exist at /Users/rahulraj/Library/Caches/ms-playwright/chromium_headless_shell-1155/chrome-mac/headless_shell
╔════════════════════════════════════════════════════════════╗
║ Looks like Playwright was just installed or updated.       ║
║ Please run the following command to download new browsers: ║
║                                                            ║
║     playwright install                                     ║
║                                                            ║
║ <3 Playwright Team                                         ║
╚════════════════════════════════════════════════════════════╝

In [ ]:
# Streaming chat loop — uses the pretty_event formatter above.
chat_questions = [
    'List every tool you have access to with a one-line purpose for each.',
    'Use web_search to find the current Python LTS version, then confirm the answer.',
]

for q in chat_questions:
    stream_and_print(q, f'CHAT: {q[:50]}')

In [ ]:
tool_rows = []
for tool in agent.tools:
    tool_rows.append({
        'name': getattr(tool, 'name', tool.__class__.__name__),
        'description': getattr(tool, 'description', ''),
        'provider': getattr(tool, 'provider', 'local'),
        'credential_key': getattr(tool, 'credential_key', ''),
    })
tool_rows[:12]

## Environment Keys

The notebook and runnable example honor these common settings:

- `SHIPIT_LLM_PROVIDER`
- `SHIPIT_BEDROCK_MODEL`
- `OPENAI_API_KEY`
- `ANTHROPIC_API_KEY`
- `GEMINI_API_KEY`
- `GROQ_API_KEY`
- `TOGETHERAI_API_KEY`
- `SHIPIT_WEB_SEARCH_PROVIDER`

For Bedrock, the default DRKCACHE-aligned model is `bedrock/openai.gpt-oss-120b-1:0`.


## Notebook Path

This file lives at `notebooks/shipit_agent_test_drive.ipynb`.
